In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics import (
    classification_report, precision_score, recall_score, f1_score, 
    confusion_matrix, roc_curve, precision_recall_curve, accuracy_score,
    mean_squared_error, r2_score
)
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

In [2]:
try:
    import shap
    print("[INFO] SHAP library found")
except ImportError:
    shap = None
    print("[INFO] SHAP library not found, skipping SHAP-related features")

print("All libraries imported successfully")

[INFO] SHAP library found
All libraries imported successfully


In [21]:
dataset_dir = r"D:\Iot Based solution\Dataset\CMAPSS Jet engine simulated data"
output_dir = "models"
os.makedirs(output_dir, exist_ok=True)

index_names = ['Engine_ID', 'Cycle']
setting_names = ['Setting_1', 'Setting_2', 'Setting_3']
sensor_names = [f'Sensor_{i}' for i in range(1, 22)]
col_names = index_names + setting_names + sensor_names

subsets = ['FD001', 'FD002', 'FD003', 'FD004']
RUL_THRESHOLD = 30 

In [26]:
for sub in subsets:
    print("\n" + "="*70)
    print(f"[PROCESS] EXECUTING PIPELINE FOR SUB-DATASET: {sub}")
    print("="*70)
    
    train_file = os.path.join(dataset_dir, f"train_{sub}.txt")
    test_file = os.path.join(dataset_dir, f"test_{sub}.txt")
    rul_file = os.path.join(dataset_dir, f"RUL_{sub}.txt")
    
    # --- EXPLICIT ERROR LOGGING FOR FILE PATHS ---
    missing_files = []
    if not os.path.exists(train_file): missing_files.append(f"train_{sub}.txt")
    if not os.path.exists(test_file): missing_files.append(f"test_{sub}.txt")
    if not os.path.exists(rul_file): missing_files.append(f"RUL_{sub}.txt")
    
    if missing_files:
        print(f"[⚠️ WARNING] Skipped {sub}! Python could not find these exact files in your directory:")
        for f in missing_files:
            print(f"  -> Expected: {os.path.join(dataset_dir, f)}")
        print("[TIP] Check if your files have names like 'train_FD001.txt' or if windows is hiding a double extension like '.txt.txt'")
        continue
    print(f"[INFO] All 3 files found. Loading logs for {sub}...")
    df_train_raw = pd.read_csv(train_file, sep=r'\s+', header=None, names=col_names)
    df_test_raw = pd.read_csv(test_file, sep=r'\s+', header=None, names=col_names)
    df_rul_ground_truth = pd.read_csv(rul_file, sep=r'\s+', header=None, names=['True_RUL'])
    df_rul_ground_truth['Engine_ID'] = df_rul_ground_truth.index + 1
    max_cycle = df_train_raw.groupby('Engine_ID')['Cycle'].max().reset_index()
    max_cycle.columns = ['Engine_ID', 'Max_Cycle']
    df_train_processed = df_train_raw.merge(max_cycle, on='Engine_ID')
    
    df_train_processed['RUL_Continuous'] = df_train_processed['Max_Cycle'] - df_train_processed['Cycle']
    df_train_processed.drop(columns=['Max_Cycle'], inplace=True)
    
    df_train_processed['RUL_Reg_Target'] = np.minimum(df_train_processed['RUL_Continuous'], 125)
    df_train_processed['RUL_Clf_Target'] = (df_train_processed['RUL_Continuous'] <= RUL_THRESHOLD).astype(int)

    # --- Process Test Targets ---
    df_test_last_cycle = df_test_raw.groupby('Engine_ID').last().reset_index()
    df_test_processed = df_test_last_cycle.merge(df_rul_ground_truth, on='Engine_ID')
    
    df_test_processed['RUL_Reg_Target'] = df_test_processed['True_RUL']
    df_test_processed['RUL_Clf_Target'] = (df_test_processed['True_RUL'] <= RUL_THRESHOLD).astype(int)

    # --- Dynamic Feature Drops ---
    features_to_drop = ['Engine_ID', 'Cycle']
    zero_variance_cols = [col for col in df_train_processed[setting_names + sensor_names].columns 
                          if df_train_processed[col].nunique() == 1]
    final_drops = list(set(features_to_drop + zero_variance_cols))
    
    X_train = df_train_processed.drop(columns=final_drops + ['RUL_Continuous', 'RUL_Reg_Target', 'RUL_Clf_Target'], errors='ignore')
    X_test = df_test_processed.drop(columns=final_drops + ['True_RUL', 'RUL_Reg_Target', 'RUL_Clf_Target'], errors='ignore')
    
    y_train_clf = df_train_processed['RUL_Clf_Target']
    y_test_clf = df_test_processed['RUL_Clf_Target']
    y_train_reg = df_train_processed['RUL_Reg_Target']
    y_test_reg = df_test_processed['RUL_Reg_Target']
    
    FEATURES_KEY = X_train.columns.tolist()

    # --- Feature Scaling ---
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --- Classifier Training ---
    print(f"[INFO] Training RandomForest CLASSIFIER for {sub}...")
    clf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
    clf_model.fit(X_train_scaled, y_train_clf)
    
    y_pred_clf = clf_model.predict(X_test_scaled)
    acc = accuracy_score(y_test_clf, y_pred_clf)
    prec = precision_score(y_test_clf, y_pred_clf, average='binary', zero_division=0)
    rec = recall_score(y_test_clf, y_pred_clf, average='binary', zero_division=0)
    f1 = f1_score(y_test_clf, y_pred_clf, average='binary', zero_division=0)
    cm = confusion_matrix(y_test_clf, y_pred_clf)

    print(f"\n[RESULT] {sub} Classification Metrics:")
    print(f"  - Accuracy  : {acc:.4f} | Precision : {prec:.4f} | Recall : {rec:.4f} | F1 : {f1:.4f}")

    # --- Regressor Training ---
    print(f"[INFO] Training RandomForest REGRESSOR for {sub}...")
    reg_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    reg_model.fit(X_train_scaled, y_train_reg)
    
    y_pred_reg = reg_model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
    r2 = r2_score(y_test_reg, y_pred_reg)
    
    print(f"[RESULT] {sub} Regression Metrics:")
    print(f"  - RMSE: {rmse:.2f} cycles | R2 Variance Score: {r2:.4f}")

    # --- Save Outputs ---
    joblib.dump(scaler, os.path.join(output_dir, f"{sub}-cmapss_scaler.joblib"))
    joblib.dump(clf_model, os.path.join(output_dir, f"{sub}-cmapss_classifier.joblib"))
    joblib.dump(reg_model, os.path.join(output_dir, f"{sub}-cmapss_regressor.joblib"))
    print(f"[INFO] Saved {sub} scalers, classifier, and regressor to '{output_dir}/'")


[PROCESS] EXECUTING PIPELINE FOR SUB-DATASET: FD001
[INFO] All 3 files found. Loading logs for FD001...
[INFO] Training RandomForest CLASSIFIER for FD001...

[RESULT] FD001 Classification Metrics:
  - Accuracy  : 0.9100 | Precision : 0.9444 | Recall : 0.6800 | F1 : 0.7907
[INFO] Training RandomForest REGRESSOR for FD001...
[RESULT] FD001 Regression Metrics:
  - RMSE: 18.19 cycles | R2 Variance Score: 0.8084
[INFO] Saved FD001 scalers, classifier, and regressor to 'models/'

[PROCESS] EXECUTING PIPELINE FOR SUB-DATASET: FD002
[INFO] All 3 files found. Loading logs for FD002...
[INFO] Training RandomForest CLASSIFIER for FD002...

[RESULT] FD002 Classification Metrics:
  - Accuracy  : 0.9768 | Precision : 0.9508 | Recall : 0.9508 | F1 : 0.9508
[INFO] Training RandomForest REGRESSOR for FD002...
[RESULT] FD002 Regression Metrics:
  - RMSE: 29.09 cycles | R2 Variance Score: 0.7073
[INFO] Saved FD002 scalers, classifier, and regressor to 'models/'

[PROCESS] EXECUTING PIPELINE FOR SUB-DATA